# 03f · 균일화 (Uniformization)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Ross 12e §6.8 (균일화, 종속 표현 $P(t)=\sum_n e^{-\nu t}\frac{(\nu t)^n}{n!}P^n$) · Norris §2.1 (Q-행렬과 지수), §2.10 (부록: 행렬 지수) · Lawler 2e §3.2 (유한 상태공간: $P_t=e^{tA}$와 $P=I+A/\lambda$) · Durrett 3e §4.2 ($e^{Qt}=\lim(I+Qt/n)^n$과 비교) |
| 선수 노트북 | 03b (콜모고로프 방정식, $e^{Qt}$, 극한 정리), 02b (푸아송 과정의 개수 분포·독립 증분) |
| 예상 소요 | 75분 (선택) |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **CTMC의 시간 구조(푸아송 알람)와 공간 구조(DTMC)를 분리할 수 있다.** 모든 상태에서 같은 비율 $\lambda$로 알람을 울리고, 울릴 때마다 확률행렬 $P=I+Q/\lambda$로 (자기 루프를 포함해) 이동하면 그 결과가 정확히 생성원 $Q$의 CTMC다. 그러면 $e^{Qt}$는 푸아송 가중치의 $P^n$ 급수가 되고(계산), CTMC 시뮬레이션은 '푸아송 개수의 DTMC 스텝'이 되어 완전히 벡터화되며(시뮬레이션), 03b에서 미뤄 둔 "CTMC에는 왜 주기가 없는가"가 자기 루프 한 줄로 설명된다(이론).

## 0. 준비 (Setup)

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import poisson
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import (is_generator, uniformize, transition_matrix, stationary_ctmc,
                        embedded_chain, gillespie_paths)
from spkit.markov import is_stochastic, stationary, simulate_chain
from spkit.poisson import poisson_arrivals
from spkit.plots import plot_hist_vs_pmf

SEED, rng = rng_for("03f")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (03e) Erlang C는 무엇의 확률이고, 거기서 $\rho$는 어떻게 정의되나?

<details><summary>정답</summary>

M/M/c에서 도착한 손님이 **기다릴** 확률 $P(N\ge c)$. $\rho=a/c=\lambda/(c\mu)$ (제공 부하 $a=\lambda/\mu$를 서버 수로 나눈 서버당 이용률).

</details>

**Q2.** (03e) Erlang B는 어떤 시스템의 무엇이고, 서비스 분포를 바꾸면 어떻게 되나?

<details><summary>정답</summary>

M/M/c/c 손실 시스템의 **차단 확률** $\dfrac{a^c/c!}{\sum_{n\le c}a^n/n!}$. 서비스 분포에 **무감각(insensitive)** — M/G/c/c에서도 같은 값이다.

</details>

**Q3.** (03e) 같은 총 용량의 빠른 서버 1대와 느린 서버 2대 중, $W$와 $W_q$는 각각 어느 쪽이 작은가?

<details><summary>정답</summary>

$W$(체류시간)는 빠른 1대가 작고, $W_q$(대기시간)는 느린 2대가 작다. 빠른 서버는 서비스 자체가 짧고, 두 대는 '둘 다 바쁠' 확률이 낮아 기다림이 짧다.

</details>

**Q4.** (03d) 리틀의 법칙 $L=\lambda W$의 표본경로 증명에서 핵심 한 줄은?

<details><summary>정답</summary>

$\int_0^T N(t)\,dt=\sum_i W_i$ — 계단함수 아래 면적을 세로(시각별 손님 수)로 세나 가로(손님별 체류시간)로 세나 같다. 양변을 $T$로 나누면 $L=(\text{도착률})\times W$.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **균일화 상수(uniformization rate)** $\lambda\ge\max_i q_i$ ($q_i=-Q_{ii}$는 체류율). spkit `uniformize(Q, lam=None)`의 기본값은 $\max_i q_i$ (등호 케이스).
- **균일화 사슬(uniformized chain)** $P=I+Q/\lambda$: $P_{ij}=q_{ij}/\lambda$ ($i\ne j$), $P_{ii}=1-q_i/\lambda$. 대각원소는 **자기 루프 = '가짜 점프(fictitious jump)'** 확률이다. 03a의 점프 사슬 $\Pi$(대각 0)와 다르다.
- **종속(subordination) 표현**: $N$은 비율 $\lambda$의 푸아송 과정, $Y$는 전이행렬 $P$의 DTMC, 서로 독립. $X_t:=Y_{N(t)}$. 말로: 모든 상태에서 같은 비율 $\lambda$로 알람이 울리고, 울릴 때 $P$로 이동한다 (자기 루프면 제자리).
- **균일화 급수**: $P(t)=e^{Qt}=\sum_{n\ge0}w_n(t)\,P^n$, $w_n(t)=e^{-\lambda t}\dfrac{(\lambda t)^n}{n!}$ — 코드로는 `scipy.stats.poisson.pmf(n, lam*t)`.
- **절단 급수(truncated series)** $P_N(t)=\sum_{n\le N}w_n(t)P^n$과 그 오차 $\|P(t)-P_N(t)\|_{\max}\le\sum_{n>N}w_n(t)=P(\mathrm{Poisson}(\lambda t)>N)$ — 코드로는 `poisson.sf(N, lam*t)`. 오차 $\varepsilon$에 필요한 $N$은 `poisson.isf(eps, lam*t)`.
- **이 노트북의 예** (03a, 03b의 기계 사슬): $Q_3=\begin{pmatrix}-1.5&1&0.5\\1&-2&1\\2&0&-2\end{pmatrix}$, $\max q_i=2\Rightarrow\lambda=2$, $P=\begin{pmatrix}0.25&0.5&0.25\\0.5&0&0.5\\1&0&0\end{pmatrix}$. $t=1$이면 $\lambda t=2$: 꼬리 $<10^{-8}$에 $N=14$항, $<10^{-12}$에 $N=18$항. $\lambda=10$으로 잡으면 $10^{-8}$에 $N=32$항.
- **점프 사슬 vs 균일화 사슬**: $\Pi$의 정상분포 $\nu$는 '방문 횟수' 분포이고 $\pi_i\propto\nu_i/q_i$로 CTMC의 $\pi$와 **다르다** (03c). 균일화 사슬 $P$의 정상분포는 $\pi$와 **같다** (정리 4).

### 2.2 정리 1 — 균일화 항등식 (Ross 12e §6.8, Norris §2.1)

> $\lambda\ge\max_i q_i$이면 $P=I+Q/\lambda$는 확률행렬이고, 모든 $t\ge0$에 대해
> $$e^{Qt}=e^{-\lambda t}\sum_{n\ge0}\frac{(\lambda t)^n}{n!}P^n .$$

**가정이 왜 필요한가**
- **$\lambda\ge\max q_i$**: $P_{ii}=1-q_i/\lambda\ge0$이어야 확률행렬이다. 급수 자체는 어떤 $\lambda>0$에서도 $e^{Qt}$로 수렴한다(순수 대수 항등식)—하지만 $P$가 확률행렬이 아니면 정리 2의 시뮬레이션 해석과 정리 3의 오차 한계가 사라진다 (E3).
- **유한 상태**: $\|P^n\|_{\max}\le1$이 항별 유계를 주고, 급수의 재배열·행렬곱과 극한의 교환이 정당화된다. $-\lambda tI$와 $\lambda tP$가 **교환**한다는 자명한 사실이 $e^{A+B}=e^Ae^B$의 조건이다 (일반 행렬에서는 이 법칙이 **거짓**이다).

<details><summary>증명</summary>

**(확률행렬)** $i\ne j$: $P_{ij}=q_{ij}/\lambda\ge0$. 대각: $P_{ii}=1-q_i/\lambda\ge1-\max_kq_k/\lambda\ge0$. 행합: $\sum_jP_{ij}=1+\frac1\lambda\sum_jQ_{ij}=1+0=1$.

**(항등식)** $Q=\lambda(P-I)$이므로 $Qt=\lambda tP-\lambda tI$. 두 행렬 $A=-\lambda tI$, $B=\lambda tP$는 교환한다($I$는 모든 행렬과 교환). 교환하는 행렬에 대해 $e^{A+B}=e^Ae^B$ (이항정리로 급수를 재배열; 절대수렴이므로 허용):
$$e^{Qt}=e^{-\lambda tI}\,e^{\lambda tP}=e^{-\lambda t}I\cdot\sum_{n\ge0}\frac{(\lambda t)^n}{n!}P^n=\sum_{n\ge0}e^{-\lambda t}\frac{(\lambda t)^n}{n!}P^n .$$
여기서 $e^{-\lambda tI}=\sum_k(-\lambda t)^kI/k!=e^{-\lambda t}I$.

**(해석)** 가중치 $w_n(t)=e^{-\lambda t}(\lambda t)^n/n!$은 $\mathrm{Poisson}(\lambda t)$ pmf이고 각 $P^n$은 확률행렬이므로, $P(t)$는 확률행렬들의 **볼록결합** $P(t)=E[P^{N(t)}]$다. $\square$

</details>

### 2.3 정리 2 — 종속 표현: 균일화 사슬로 시뮬레이션 (Ross 12e §6.8)

> $N\sim$ 푸아송 과정($\lambda$), $Y\sim$ DTMC($P$), 서로 독립이면 $X_t:=Y_{N(t)}$는 생성원 $Q$의 CTMC다. 특히 $P(X_t=j\mid X_0=i)=(e^{Qt})_{ij}$이고, $X$가 상태 $i$에 (진짜로) 머무는 시간은 $\mathrm{Exp}(q_i)$다.

**가정이 왜 필요한가**
- **$N$과 $Y$의 독립**: 스텝 수와 스텝 내용이 얽히면 $P(X_t=j)=\sum_nP(N(t)=n)\,(P^n)_{ij}$ 분해가 안 된다.
- **$N$의 독립 정상 증분** (02b): 시각 $s$ 이후의 알람 개수가 과거와 독립이어야 마르코프 성질이 $X$에 전달된다.

**증명 스케치.** (주변분포) $N(t)$와 $Y$의 독립으로
$P(X_t=j\mid X_0=i)=\sum_nP(N(t)=n)\,P(Y_n=j\mid Y_0=i)=\sum_nw_n(t)(P^n)_{ij}=(e^{Qt})_{ij}$ (정리 1).
(마르코프성) 시각 $s$ 이후의 알람은 $[0,s]$의 알람과 독립인 푸아송 과정이고 $Y$는 마르코프이므로, $X_s$가 주어지면 $(X_{s+u})_{u\ge0}$는 $X_s$에서 출발한 같은 구성이다 — Ross 12e §6.8.
(체류시간) 상태 $i$에서 알람은 $\mathrm{Exp}(\lambda)$ 간격으로 울리고, 각 알람이 '진짜 이동'일 확률은 $1-P_{ii}=q_i/\lambda=:p$로 알람 시각과 독립. 진짜 이동까지 알람 수 $G\sim\mathrm{Geom}(p)$이고 체류시간 $H=\sum_{k\le G}E_k$, $E_k\sim\mathrm{Exp}(\lambda)$. 라플라스 변환 $E[e^{-sH}]=\sum_{g\ge1}(1-p)^{g-1}p\left(\frac{\lambda}{\lambda+s}\right)^g=\frac{p\lambda}{p\lambda+s}$, 즉 $H\sim\mathrm{Exp}(p\lambda)=\mathrm{Exp}(q_i)$ (03d 정리 4의 계산과 같은 구조; E2에서 유도·수치 확인). 진짜 이동의 목적지는 $\frac{q_{ij}/\lambda}{q_i/\lambda}=q_{ij}/q_i=\Pi_{ij}$ — 03a의 점프 사슬 그대로. $\square$

### 2.4 정리 3 — 절단 오차 한계

> $$\max_{i,j}\Big|P_{ij}(t)-\sum_{n\le N}w_n(t)(P^n)_{ij}\Big|\;\le\;\sum_{n>N}w_n(t)=P\big(\mathrm{Poisson}(\lambda t)>N\big).$$

**가정이 왜 필요한가**
- **$P^n$이 확률행렬** ($0\le(P^n)_{ij}\le1$): 꼬리의 각 항을 가중치 $w_n$ 하나로 누르는 유일한 근거다. $\lambda<\max q_i$면 $|(P^n)_{ij}|$가 지수적으로 커져 한계가 무너진다 (E3: $\max|P^{40}|\approx8\times10^{10}$).

<details><summary>증명</summary>

$P_{ij}(t)-P_{N,ij}(t)=\sum_{n>N}w_n(t)(P^n)_{ij}$. 절댓값을 취하고 삼각부등식: $\le\sum_{n>N}w_n(t)\,|(P^n)_{ij}|\le\sum_{n>N}w_n(t)\cdot1$. 마지막 합은 $\mathrm{Poisson}(\lambda t)$가 $N$을 넘을 확률이다. $\square$

</details>

### 2.5 정리 4 — CTMC의 무주기성과 수렴: 03b 정리 3의 완성 (Lawler 2e §3.2, Durrett 3e §4.2)

> $\lambda>\max_iq_i$로 잡으면 $P=I+Q/\lambda$는 모든 대각원소가 양수라 **비주기적**이고, $Q$가 기약이면 $P$도 기약이다. 따라서 01e의 수렴 정리로 $P^n\to\mathbf1\pi$이며, $\pi P=\pi\iff\pi Q=0$이므로 **$P$의 정상분포는 CTMC의 $\pi$와 같다.** 결과적으로 $P(t)=\sum_nw_n(t)P^n\to\mathbf1\pi$ ($t\to\infty$).

**가정이 왜 필요한가**
- **$\lambda>\max q_i$ (등호 아님)**: 등호면 $q_i=\lambda$인 상태의 자기 루프가 0이다. 등호 케이스에서는 $q_i<\lambda$인 상태가 **하나라도** 있어야 (기약성과 함께) 그 자기 루프로 비주기성이 보장된다. 모든 $q_i$가 같으면 $P$는 주기적일 수 있다: $Q=\begin{pmatrix}-1&1\\1&-1\end{pmatrix}$, $\lambda=1$이면 $P=\begin{pmatrix}0&1\\1&0\end{pmatrix}$이라 $P^n$은 진동하고 $P^n\to\mathbf1\pi$가 **거짓**이다. 그래도 $\pi P=\pi\iff\pi Q=0$과 $P(t)\to\mathbf1\pi$ (03b 정리 3)는 등호 케이스에서도 성립한다 — 자기 루프가 없어도 푸아송 개수 $N(t)$가 홀짝을 섞어 주기 때문이다. 엄격한 부등호($\lambda>\max q_i$)로 잡으면 이 모든 경우 구분 없이 논증이 한 줄이다.
- **기약성**: 정상분포의 유일성과 초기 상태와 무관한 수렴에 필요하다.

<details><summary>증명</summary>

**(정상분포)** $\pi P=\pi(I+Q/\lambda)=\pi+\pi Q/\lambda$이므로 $\pi P=\pi\iff\pi Q=0$. 즉 두 방정식의 해집합이 같고, 정규화 $\sum\pi_i=1$까지 같으므로 정상분포가 같다.

**(비주기·기약)** $P_{ii}=1-q_i/\lambda>0$이므로 모든 상태에 길이 1의 순환이 있어 주기 1. $i\ne j$에 대해 $P_{ij}>0\iff q_{ij}>0$이므로 $P$의 그래프는 $Q$의 그래프와 같다(자기 루프 제외) — $Q$가 기약이면 $P$도 기약.

**(수렴)** 01e 수렴 정리: 기약·비주기 유한 DTMC에서 $\forall\varepsilon>0\;\exists n_0$: $n\ge n_0\Rightarrow\|P^n_{i\cdot}-\pi\|_{TV}<\varepsilon$ (모든 $i$). 정리 1의 볼록결합으로
$$\|P_{i\cdot}(t)-\pi\|_{TV}\le\sum_{n}w_n(t)\,\|P^n_{i\cdot}-\pi\|_{TV}\le\sum_{n<n_0}w_n(t)\cdot1+\varepsilon .$$
첫 항은 $P(\mathrm{Poisson}(\lambda t)<n_0)\to0$ ($t\to\infty$; 푸아송 질량이 평균 $\lambda t$ 근처로 옮겨 간다). $\varepsilon$이 임의이므로 $P_{i\cdot}(t)\to\pi$. $\square$

</details>

### 2.6 직관

**'빠른 시계 하나 + 가짜 점프.'** 03a의 구성은 상태마다 속도가 다른 알람($q_i$)을 썼다. 균일화는 가장 빠른 속도 $\lambda$의 알람 하나를 모든 상태에서 똑같이 울리되, 실제로는 확률 $q_i/\lambda$로만 움직이고 나머지는 제자리(가짜 점프)에 둔다. 그러면 **언제**(푸아송 과정)와 **어디로**(DTMC)가 독립으로 분리되어 $P(t)=E[P^{N(t)}]$가 된다.

03b의 $e^{Qt}=\lim_n(I+Qt/n)^n$은 '고정 개수 $n$의 아주 작은 스텝'이었고 극한을 취해야 정확했다. 균일화는 '**푸아송 개수**의 유한 스텝'이라 극한 없이 정확하다. 비용은 $\lambda$를 크게 잡을수록 가짜 점프가 늘어 필요한 항 수가 $\lambda t+O(\sqrt{\lambda t})$로 커지는 것이다. 큰 희소 $Q$에서는 벡터곱 $vP^n$만 있으면 되므로 `expm`(밀집 행렬)보다 유리하고, 시뮬레이션에서는 질레스피(경로별 파이썬 루프)와 달리 `simulate_chain` 한 번으로 모든 경로를 벡터화할 수 있다.

### 2.7 함정(traps)

1. $\lambda<\max q_i$면 $P$의 대각이 음수 — 확률행렬이 아니라 시뮬레이션이 무의미하고 오차 한계가 무효다. 급수는 여전히 $e^{Qt}$로 수렴하지만 항 크기가 폭발해 상쇄 오차 위험이 있다 (E3).
2. $\lambda$가 크면 필요한 항 수 $\approx\lambda t+c\sqrt{\lambda t}$가 는다. 큰 $t$의 분포는 급수 대신 $\pi$(03c)나 `expm`을 쓰라.
3. 균일화 사슬 $P$의 정상분포는 $\pi$와 같지만 점프 사슬 $\Pi$의 정상분포 $\nu$는 다르다 (03c 트랩). 4.1절에서 둘을 나란히 출력한다.
4. 푸아송 가중치는 `scipy.stats.poisson.pmf`로 — `np.exp(-x)*x**n/factorial(n)`은 큰 $n$에서 오버플로한다.
5. 벡터화 시뮬레이션에서 경로별 최종 상태는 `X[np.arange(n_paths), N_i]`(경로마다 **다른** 열)이지 마지막 열 `X[:, -1]`이 아니다. 마지막 열은 모든 경로를 $\max N_i$ 스텝까지 굴린 것이라 $t$보다 먼 미래다.
6. `uniformize(Q)`의 기본 $\lambda=\max q_i$는 등호 케이스라 $q_i=\lambda$인 상태의 자기 루프가 0이다. 정리 4의 논증에는 엄격한 부등호가 편하지만, 결과 중 정상분포($\pi P=\pi$)와 $P(t)\to\mathbf1\pi$는 등호 케이스에서도 그대로다. 깨질 수 있는 것은 **$P^n$의 수렴**뿐이다 — 모든 $q_i$가 같으면 $P$에 자기 루프가 하나도 없어 주기적일 수 있다 (2.5절의 2상태 예). $Q_3$는 $q_0=1.5<2$라 상태 0의 루프가 남아 문제없다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

- (a) 왜 $\lambda<\max q_i$면 균일화 사슬이 확률행렬이 아닌지 한 줄로.
- (b) 점프 사슬 $\Pi$와 균일화 사슬 $P$의 차이를 **정상분포 관점**에서 설명해 보세요 ($\nu$ vs $\pi$, 그리고 $\pi_i\propto\nu_i/q_i$가 나오는 이유).
- (c) 정리 1의 증명에서 "교환한다"가 어디에 쓰였는지 짚어 보세요.

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향이 맞는지가 목적입니다. 힌트: $Q_3$의 체류율은 $q=(1.5,2,2)$, $\pi=(1/2,1/4,1/4)$이고, $P_{02}(1)$은 03b에서 이미 `expm`으로 본 값입니다.

In [ ]:
predictions = {
    # Q3 를 lambda=2 로 균일화하면 상태 0 의 자기 루프(가짜 점프) 확률 P_00 은? (2.1 절의 정의를 찾아보세요)
    "p_self_loop_0": None,
    # 균일화 급수로 계산한 P_02(1) 은 03b 의 expm 값과 같아야 한다. 그 값은?
    "p02_at_1": None,
    # lambda t = 2 에서 절단 오차 한계 P(Poisson(2) > N) 을 1e-8 아래로 만드는 최소 항 수 N 은? (정수)
    "n_terms_1e8": None,
    # 균일화 시뮬레이션에서 t=1 까지의 DTMC 스텝 수(가짜 점프 포함)의 기댓값은?
    "mean_steps_t1": None,
    # 균일화 사슬 P 의 정상분포의 첫 성분은? (Q3 의 pi_0 와 같은가, 다른가? 값을 적으세요)
    "pi0_unif": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 균일화 사슬 만들기와 정상분포 (단계 1)

`uniformize(Q3)`는 기본값 $\lambda=\max q_i=2$로 $P=I+Q/\lambda$를 만든다. 확인할 것: 확률행렬인가, 정상분포가 $\pi$와 같은가, 그리고 점프 사슬 $\Pi$의 정상분포 $\nu$는 왜 다른가 (함정 3).

In [ ]:
Q3 = np.array([[-1.5, 1.0, 0.5],
               [ 1.0, -2.0, 1.0],
               [ 2.0, 0.0, -2.0]])          # 0=정상, 1=성능저하, 2=고장 (03a)
assert is_generator(Q3)

P, lam = uniformize(Q3)                       # 기본 lam = max q_i
print(f"lam = {lam},  is_stochastic(P) = {is_stochastic(P)}")
print("P = I + Q/lam =\n", P)
assert is_stochastic(P) and np.isclose(lam, 2.0)

pi = stationary_ctmc(Q3)                      # pi Q = 0
pi_unif = stationary(P)                       # pi P = pi
np.testing.assert_allclose(pi_unif, pi, atol=1e-10)
Pi, q = embedded_chain(Q3)                    # 점프 사슬 (대각 0), 체류율 q
nu = stationary(Pi)
print(f"pi  (CTMC, pi Q = 0)        = {pi}")
print(f"stationary(P_uniformized)   = {pi_unif}   <- same as pi (Theorem 4)")
print(f"stationary(Pi_jump)  = nu   = {nu.round(6)}   = (3/7, 2/7, 2/7) <- different!")
print(f"nu_i / q_i, normalised      = {(nu / q / (nu / q).sum()).round(6)}   <- pi again (03c)")

균일화 사슬의 대각 $(0.25,0,0)$: 상태 1, 2는 $q_i=\lambda$라 자기 루프가 0이고 상태 0만 확률 $0.25$의 가짜 점프를 갖는다 (등호 케이스, 함정 6 — 루프가 하나는 남았으므로 기약인 $P$는 비주기적이다). 정상분포는 $\pi$와 정확히 같고, 점프 사슬의 $\nu$는 '방문 횟수' 분포라 다르다.

정리 2를 그림으로 먼저 본다. 경로 하나: 비율 $\lambda$의 알람 시각 $N$ (`poisson_arrivals`), 알람마다 $P$로 한 스텝 (`simulate_chain`), $X_t=Y_{N(t)}$. 진짜 점프와 가짜 점프(자기 루프)를 구분해 표시한다 — 가짜 점프는 상태 0에서만 나온다 ($P_{00}=0.25$, 나머지 대각은 0).

In [ ]:
T_show = 6.0
alarms = poisson_arrivals(lam, T_show, rng)                 # 알람 시각 (비율 lam)
Y = simulate_chain(P, 0, len(alarms), rng)[0]              # 알람마다 P 로 한 스텝
t_path = np.concatenate([[0.0], alarms, [T_show]])
x_path = np.concatenate([Y, [Y[-1]]])
real = Y[1:] != Y[:-1]                                     # 상태가 바뀐 알람 = 진짜 점프

fig, ax = plt.subplots()
ax.step(t_path, x_path, where="post", lw=1.5, color="C0", label="X_t = Y_{N(t)}")
ax.plot(alarms[real], Y[1:][real], "o", color="C3", label=f"alarm -> real jump ({real.sum()})")
ax.plot(alarms[~real], Y[1:][~real], "x", color="C2", ms=8, label=f"alarm -> fictitious jump ({(~real).sum()})")
ax.set_yticks([0, 1, 2], ["0 up", "1 degraded", "2 down"]); ax.set_xlabel("t")
ax.set_title(f"Fig 1: One uniformized path of the machine chain (lambda = {lam:g}, {len(alarms)} alarms)")
ax.legend(fontsize=8)
plt.show()

### 4.2 균일화 급수 vs `expm`, 그리고 $\lambda$ 무관성 (단계 2)

절단 차수는 `poisson.isf(tol, lam*t)`로 정한다 (꼬리가 `tol` 아래로 떨어지는 최소 $N$). 급수 계산 자체는 spkit에 없으므로 여기서 정의한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def n_terms(lam, t, tol=1e-12):
    """Poisson(lam t) 꼬리 P(N > n) 이 tol 아래로 떨어지는 최소 절단 차수 N."""
    return int(poisson.isf(tol, lam * t)) if lam * t > 0 else 0

def uniformized_P(Q, t, lam=None, tol=1e-12, N=None):
    """균일화 급수 sum_{n<=N} w_n(t) P^n 으로 P(t) 를 계산한다. lam=None 이면 max q_i."""
    Q = np.asarray(Q, float); n = len(Q)
    if lam is None:
        lam = float(-np.diag(Q).min())        # max holding rate
    P = np.eye(n) + Q / lam
    if N is None:
        N = n_terms(lam, t, tol)
    w = poisson.pmf(np.arange(N + 1), lam * t)   # 함정 4: factorial 대신 scipy pmf
    out = np.zeros((n, n)); Pn = np.eye(n)
    for k in range(N + 1):
        out += w[k] * Pn
        Pn = Pn @ P
    return out

In [ ]:
E1 = transition_matrix(Q3, 1.0)                     # expm 참조값 (03b)
S2 = uniformized_P(Q3, 1.0, lam=2.0)
S10 = uniformized_P(Q3, 1.0, lam=10.0)
err2, err10 = np.abs(S2 - E1).max(), np.abs(S10 - E1).max()
print(f"|series(lam=2)  - expm|_max = {err2:.2e}   (truncated at N = {n_terms(2.0, 1.0)}, tol=1e-12)")
print(f"|series(lam=10) - expm|_max = {err10:.2e}   (truncated at N = {n_terms(10.0, 1.0)}, tol=1e-12)")
print(f"|series(lam=10) - series(lam=2)|_max = {np.abs(S10 - S2).max():.2e}   <- lambda-independence")
assert err2 < 1e-10 and err10 < 1e-10
assert np.abs(S10 - S2).max() < 1e-10

print("\ntruncation order N for a tail bound < eps at t = 1:")
for lam_ in (2.0, 10.0):
    print(f"  lam = {lam_:>4g}: " + ",  ".join(f"eps={eps:.0e} -> N={n_terms(lam_, 1.0, eps)}" for eps in (1e-8, 1e-12)))
print(f"\nP_02(1) series = {S2[0, 2]:.10f},  expm = {E1[0, 2]:.10f}")

In [ ]:
n_axis = np.arange(0, 45)
fig, ax = plt.subplots()
for lam_, c, dx in ((2.0, "C0", -0.15), (10.0, "C1", 0.15)):
    N8 = n_terms(lam_, 1.0, 1e-8)
    ax.stem(n_axis + dx, poisson.pmf(n_axis, lam_), linefmt=c + "-", markerfmt=c + "o", basefmt=" ",
            label=f"lambda = {lam_:g}: N = {N8} terms for tail < 1e-8")
    ax.axvline(N8, color=c, ls=":", lw=1)
ax.set_xlabel("n (number of DTMC steps by t = 1)"); ax.set_ylabel("w_n(1) = Poisson(lambda) pmf")
ax.set_title("Fig 2: Poisson(lambda t) weights of the uniformization series at t = 1")
ax.legend()
plt.show()

$\lambda=2$와 $\lambda=10$의 급수는 같은 행렬 $e^{Q}$를 준다 — 차이는 $10^{-14}$ 수준. 달라지는 것은 가중치의 위치(평균 $\lambda t$)와 폭($\sqrt{\lambda t}$), 즉 필요한 항 수(14 vs 32)와 $P$의 자기 루프 확률뿐이다 (함정 2).

### 4.3 절단 오차 vs 푸아송 꼬리 한계 (단계 3)

정리 3: $N=0,\dots,20$에서 실제 오차 $\max|P_N(1)-e^{Q}|$와 한계 $P(\mathrm{Poisson}(2)>N)$를 겹쳐 그린다. 오차는 **항상** 한계 아래에 있어야 한다.

In [ ]:
N_grid = np.arange(0, 21)
err = np.array([np.abs(uniformized_P(Q3, 1.0, lam=2.0, N=N) - E1).max() for N in N_grid])
bound = poisson.sf(N_grid, lam * 1.0)              # P(Poisson(2) > N)
assert (err <= bound + 1e-13).all()
for N in (0, 5, 10, 14, 20):
    print(f"N = {N:>2d}: error {err[N]:.2e}   bound {bound[N]:.2e}")

fig, ax = plt.subplots()
ax.semilogy(N_grid, bound, "k--", label="bound P(Poisson(2) > N)")
ax.semilogy(N_grid, err, "C0o-", label="actual max |P_N(1) - expm(Q)|")
ax.axhline(1e-8, color="C3", ls=":", lw=1, label="1e-8 target")
ax.set_ylim(1e-16, 2); ax.set_xlabel("truncation order N"); ax.set_ylabel("max abs error")
ax.set_title("Fig 3: Truncation error of the uniformization series vs the Poisson tail bound")
ax.legend()
plt.show()

한계는 보수적이지만 자릿수는 정확히 따라간다 (오차/한계 $\approx0.5$ — $P^n$의 원소가 1이 아니라 $\le1$이기 때문에 생기는 여유). $N=20$에서 오차 $\sim4\times10^{-15}$는 이미 배정도 부동소수점 바닥 근처이므로 그 이상은 의미가 없다.

### 4.4 균일화 사슬로 CTMC 시뮬레이션 (단계 4)

정리 2를 측정으로 옮긴다. 4.1의 경로 하나를 $N_{\text{paths}}$개 경로로, **한꺼번에**: 각 경로의 스텝 수 $N_i\sim\mathrm{Poisson}(\lambda t)$를 뽑고, 모든 경로를 $\max_iN_i$ 스텝까지 굴린 뒤 경로 $i$의 $N_i$번째 열을 읽는다 (함정 5). 질레스피(경로별 루프)와 벽시계 시간을 비교한다.

In [ ]:
t1 = 1.0
tic = time.perf_counter()
N_i = rng.poisson(lam * t1, N_PATHS)                                   # 경로별 스텝 수
X = simulate_chain(P, 0, int(N_i.max()), rng, n_paths=N_PATHS)        # (N_PATHS, max N_i + 1)
final = X[np.arange(N_PATHS), N_i]                                     # 경로마다 다른 열! (X[:, -1] 아님)
t_unif = time.perf_counter() - tic

tic = time.perf_counter()
_, Xg = gillespie_paths(Q3, 0, t1, rng, N_PATHS, n_grid=2)             # 비교용: 03a 질레스피
t_gil = time.perf_counter() - tic

est_p2, est_p0 = mc_proportion(final == 2), mc_proportion(final == 0)
est_steps = mc_estimate(N_i)
est_p2_gil = mc_proportion(Xg[:, -1] == 2)
print(f"uniformized (vectorised): {t_unif * 1e3:7.1f} ms   P(X_1 = 2) = {est_p2}")
print(f"Gillespie (per-path loop): {t_gil * 1e3:7.1f} ms   P(X_1 = 2) = {est_p2_gil}")
print(f"P(X_1 = 0) = {est_p0},   mean steps N_i = {est_steps},   max N_i = {N_i.max()}")
print(f"wrong column X[:, -1]: P(X = 2) = {(X[:, -1] == 2).mean():.4f}  (that is ~t = {N_i.max() / lam:.1f}, not t = 1)")

n_sup = np.arange(0, N_i.max() + 1)
ax = plot_hist_vs_pmf(N_i, n_sup, poisson.pmf(n_sup, lam * t1), label_pmf="Poisson(2) pmf")
ax.set_xlabel("number of uniformized DTMC steps N(1)"); ax.set_ylabel("density")
ax.set_title(f"Fig 4: Number of uniformized DTMC steps by t = 1 (n = {N_PATHS}) vs Poisson(2) pmf")
plt.show()

경로 수가 같아도 균일화는 numpy 배열 연산 몇 번(스텝 수 $\le\max N_i\approx10$)이고, 질레스피는 경로마다 파이썬 루프를 돈다. 두 방법 모두 $P(X_1=2)$를 같은 값 근처로 추정해야 한다 — 5절에서 `expm`과 4SE 기준으로 판정한다. 잘못된 열 `X[:, -1]`은 $t\approx\max N_i/\lambda$ 시각의 분포라 이미 $\pi_2=0.25$에 가깝다.

### 4.5 곡선 겹치기: $t\in[0,4]$에서 급수 vs `expm`, 항 수 $N(t)$ (단계 5)

In [ ]:
t_grid = np.linspace(0, 4, 41)
series_curve = np.array([uniformized_P(Q3, t)[0] for t in t_grid])       # (41, 3): P_0.(t) 급수
expm_curve = np.array([transition_matrix(Q3, t)[0] for t in t_grid])     # (41, 3): P_0.(t) expm
N_t = np.array([n_terms(lam, t) for t in t_grid])                        # tol = 1e-12 에 필요한 항 수
max_diff = np.abs(series_curve - expm_curve).max()
print(f"max |series - expm| over t in [0, 4]: {max_diff:.2e}")
assert max_diff < 1e-10
print(f"terms N(t) at t = 0, 1, 2, 4: {N_t[[0, 10, 20, 40]]}   (~ lam t + c sqrt(lam t))")

fig, ax = plt.subplots()
names = ["0 (up)", "1 (degraded)", "2 (down)"]
for j in range(3):
    ax.plot(t_grid, expm_curve[:, j], color=f"C{j}", label=f"expm: P_0{j}(t) -> {names[j]}")
    ax.plot(t_grid, series_curve[:, j], "o", color=f"C{j}", ms=4, mfc="none")
ax.plot([], [], "ko", mfc="none", ms=4, label="uniformization series (markers)")
ax.set_xlabel("t"); ax.set_ylabel("probability"); ax.legend(loc="center right", fontsize=8)
ax2 = ax.twinx()
ax2.step(t_grid, N_t, where="post", color="gray", lw=1, alpha=0.7)
ax2.set_ylabel("series terms N(t) for tail < 1e-12", color="gray")
ax.set_title("Fig 5: P_0j(t) via uniformization series (markers) vs expm (lines); N(t) on right axis")
plt.show()

마커가 선 위에 정확히 얹힌다 (최대 차이 $\sim10^{-13}$). 회색 계단은 항 수 $N(t)$: $t$에 대해 대략 선형($\lambda t$)에 $\sqrt{\lambda t}$ 폭의 여유가 붙어 늘어난다 — 함정 2의 비용이다. 큰 $t$라면 $P(t)\approx\mathbf1\pi$ (정리 4)를 쓰는 편이 싸다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
p00_exact, _, p02_exact = transition_matrix(Q3, 1.0)[0]           # expm 참조값
p_self_loop_exact = 1.0 - (-Q3[0, 0]) / lam                       # 1 - q_0/lam
mean_steps_exact = lam * t1                                       # N(1) ~ Poisson(lam t)
N_min = 0                                                         # 꼬리 < 1e-8 인 최소 N (직접 탐색)
while poisson.sf(N_min, lam * 1.0) >= 1e-8:
    N_min += 1

rows = [
    {"name": "P_02(1): down at t=1 [uniformized sim vs expm]", "predicted": predictions["p02_at_1"], "estimate": est_p2, "exact": p02_exact},
    {"name": "P_00(1): up at t=1 [uniformized sim vs expm]", "predicted": None, "estimate": est_p0, "exact": p00_exact},
    {"name": "E[N(1)]: mean DTMC steps by t=1 [sim vs lam t]", "predicted": predictions["mean_steps_t1"], "estimate": est_steps, "exact": mean_steps_exact},
    {"name": "P_02(1) [Gillespie sim vs expm]", "predicted": None, "estimate": est_p2_gil, "exact": p02_exact},
    {"name": "P_00 self-loop [uniformize() vs 1 - q_0/lam]", "predicted": predictions["p_self_loop_0"], "estimate": float(P[0, 0]), "exact": p_self_loop_exact},
    {"name": "N for tail < 1e-8 [poisson.isf vs search]", "predicted": predictions["n_terms_1e8"], "estimate": float(n_terms(lam, 1.0, 1e-8)), "exact": float(N_min)},
    {"name": "pi_0 of uniformized chain [stationary(P) vs pi(Q3)]", "predicted": predictions["pi0_unif"], "estimate": float(pi_unif[0]), "exact": float(pi[0])},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **P_02(1), P_00(1) (균일화 시뮬레이션)**: 정리 2가 틀렸거나 — 훨씬 흔하게 — 최종 상태를 `X[:, -1]`로 읽었다 (함정 5).
- **E[N(1)]**: `rng.poisson(lam * t)`의 인수가 $\lambda t$가 아니거나 $\lambda$가 `uniformize`의 반환값과 다르다.
- **P_02(1) (질레스피)**: 03a의 알람 시계 구성과 `expm`이 어긋난다는 뜻 — 03b 정리 2를 의심한다.
- **자기 루프 P_00**: `uniformize`의 $\lambda$ 또는 $P=I+Q/\lambda$의 부호를 확인한다.
- **항 수 N**: `poisson.isf`는 꼬리 $P(X>N)$ 기준이므로 `sf`와 정의가 맞아야 한다 (한 칸 어긋남 = 부등호 방향 착오).
- **π₀**: 정리 4의 $\pi P=\pi\iff\pi Q=0$이 깨진다는 뜻 — `stationary`에 $\Pi$를 넣은 것은 아닌지.

In [ ]:
assert_close(est_p2, p02_exact, k=4, name="p02_at_1")
assert_close(est_p0, p00_exact, k=4, name="p00_at_1")
assert_close(est_steps, mean_steps_exact, k=4, name="mean_steps_t1")
assert_close(est_p2_gil, p02_exact, k=4, name="p02_at_1 (Gillespie)")
print("all Monte Carlo checks passed (4 SE)")

## 6. 연습문제

### E1 계산

손으로: 2상태 사슬($\alpha=1,\beta=2$, 즉 $Q_2=\begin{pmatrix}-1&1\\2&-2\end{pmatrix}$)을 $\lambda=2$로 균일화한 $P$와 $P^2$를 쓰고, $t=0.5$에서 급수를 3항($n\le2$)까지만 더한 $P_{01}(0.5)$의 근사값과 정리 3의 오차 한계를 구하라. 03a의 정확값 $P_{01}(0.5)=\frac13(1-e^{-1.5})=0.258957$과 비교하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$P=I+Q_2/2=\begin{pmatrix}0.5&0.5\\1&0\end{pmatrix}$, $P^2=\begin{pmatrix}0.75&0.25\\0.5&0.5\end{pmatrix}$. $\lambda t=1$이므로 $w=(e^{-1},e^{-1},e^{-1}/2)$.
$$P_{01}(0.5)\approx e^{-1}\big(0+0.5+\tfrac12\cdot0.25\big)=0.625\,e^{-1}=0.229925 .$$
한계: $P(\mathrm{Poisson}(1)>2)=1-e^{-1}(1+1+\tfrac12)=0.080301$. 실제 오차: $0.258957-0.229925=0.029032\le0.080301$ ✓. (급수는 항상 아래에서 접근한다 — 모든 항이 음이 아니기 때문이다.)

```python
Q2 = np.array([[-1.0, 1.0], [2.0, -2.0]])
P2u, lam2 = uniformize(Q2, lam=2.0)
w = poisson.pmf([0, 1, 2], lam2 * 0.5)
approx = (w[0] * np.eye(2) + w[1] * P2u + w[2] * P2u @ P2u)[0, 1]
exact = transition_matrix(Q2, 0.5)[0, 1]                 # = (1/3)(1 - e^{-1.5})
print(P2u @ P2u, approx, exact, exact - approx, poisson.sf(2, lam2 * 0.5))
# -> [[0.75 0.25] [0.5 0.5]]  0.229925  0.258957  0.029032  0.080301
```

</details>

### E2 유도 후 시뮬레이션 검증

균일화 사슬에서 상태 $i$의 '진짜' 체류시간이 $\mathrm{Exp}(q_i)$임을 유도하라 (기하 개수의 $\mathrm{Exp}(\lambda)$ 합). 그 다음 $Q_3$의 상태 1($q_1=2$)을 $\lambda=4$로 균일화해 `G = rng.geometric(q_1 / lam, n)`, 각 경로에 `rng.exponential(1 / lam, G_i).sum()`으로 체류시간을 만들어 평균 $1/2$, 분산 $1/4$와 비교하고 (`assert_close(mean, 0.5, k=4)`), 히스토그램을 $\mathrm{Exp}(2)$ pdf와 겹쳐라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** $p=q_i/\lambda$. 알람 간격 $E_k\sim\mathrm{Exp}(\lambda)$ i.i.d., 진짜 이동까지 알람 수 $G\sim\mathrm{Geom}(p)$ ($P(G=g)=(1-p)^{g-1}p$), $G$와 $(E_k)$ 독립. $H=\sum_{k=1}^GE_k$의 라플라스 변환:
$$E[e^{-sH}]=\sum_{g\ge1}(1-p)^{g-1}p\Big(\frac{\lambda}{\lambda+s}\Big)^g=\frac{p\lambda}{\lambda+s}\cdot\frac{1}{1-(1-p)\frac{\lambda}{\lambda+s}}=\frac{p\lambda}{\lambda+s-(1-p)\lambda}=\frac{p\lambda}{p\lambda+s}.$$
이것은 $\mathrm{Exp}(p\lambda)=\mathrm{Exp}(q_i)$의 라플라스 변환이다. $\lambda$가 사라진다는 점이 핵심 — 가짜 점프를 얼마나 많이 넣든 진짜 체류시간의 분포는 같다. $q_1=2$: 평균 $1/2$, 분산 $1/4$.

```python
q1, lam4 = 2.0, 4.0
n = scale(20_000)
G = rng.geometric(q1 / lam4, n)                              # 진짜 이동까지의 알람 수
H = np.array([rng.exponential(1 / lam4, g).sum() for g in G])  # 기하 개수의 Exp(lam) 합
est_H = mc_estimate(H)
print(f"mean {est_H} (exact 0.5),  var {H.var(ddof=1):.4f} (exact 0.25),  mean G = {G.mean():.3f} (exact 2)")
assert_close(est_H, 1 / q1, k=4, name="E2 holding time mean")

x = np.linspace(0, 4, 200)
fig, ax = plt.subplots()
ax.hist(H, bins=60, density=True, alpha=0.5, label="geometric sum of Exp(4) gaps")
ax.plot(x, q1 * np.exp(-q1 * x), "k-", label="Exp(2) pdf")
ax.set_xlabel("holding time in state 1"); ax.set_ylabel("density"); ax.legend()
ax.set_title("E2: real holding time of the uniformized chain is Exp(q_1)")
plt.show()
# 20000 개에서 평균 0.498 ± 0.0035, 분산 0.246 (같은 계산을 rng.gamma(G, 1/lam4) 로 한 번에 할 수도 있다)
```

</details>

### E3 가정을 깨보기

$\lambda=1<\max q_i=2$로 $P=I+Q_3$을 만들어 (`uniformize(Q3, lam=1.0)`은 `ValueError`를 내므로 직접 만든다) (a) $P$의 대각원소, (b) `is_stochastic(P)`, (c) 급수 $\sum_{n\le N}w_n(1)P^n$이 $N=10,20$에서 `expm`과 얼마나 다른지, (d) $\max|P^N|$의 성장($N=10,20,40$)을 보고하라. `simulate_chain(P, 0, 10, rng, n_paths=5)`를 부르면 어떻게 되는가? 어떤 가정이 깨졌는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 대각 $(-0.5,-1,-1)$ — 음의 '확률'. (b) `False`. (c) 급수는 **대수 항등식**이라 여전히 $e^{Q_3}$로 수렴한다: $N=10$ 오차 $8.0\times10^{-6}$, $N=20$ 오차 $\sim10^{-15}$ (이 작은 예에서는 상쇄가 아직 심하지 않다). 그러나 정리 3의 한계 $P(\mathrm{Poisson}(1)>10)=1.0\times10^{-8}$은 실제 오차 $8\times10^{-6}$보다 **작다** — 한계가 무효다. (d) $\max|P^{10}|\approx460$, $\max|P^{20}|\approx2.9\times10^5$, $\max|P^{40}|\approx8.1\times10^{10}$: 항이 지수적으로 커지므로 $t$가 커지면 (가중치 $w_n$이 큰 $n$까지 퍼지면) 거대한 항들의 상쇄로 배정도 정밀도가 무너진다.

`simulate_chain`은 검사 없이 누적합 `C = P.cumsum(1)`과 `u > C`를 비교하므로 예외 없이 **무의미한** 경로를 만든다: 행 1의 누적합은 $(1,0,1)$이라 `u > C` 개수가 항상 1 — 상태 1이 흡수 상태처럼 동작한다. 깨진 가정: $\lambda\ge\max q_i$ (확률행렬 조건). 급수의 수렴은 살아남지만 확률 해석(정리 2)과 오차 한계(정리 3)는 죽는다.

```python
P_bad = np.eye(3) + Q3 / 1.0                                 # lam = 1 < max q_i = 2
print("diag:", np.diag(P_bad), " is_stochastic:", is_stochastic(P_bad))
for N in (10, 20):
    err_bad = np.abs(uniformized_P(Q3, 1.0, lam=1.0, N=N) - transition_matrix(Q3, 1.0)).max()
    print(f"N = {N}: |series - expm| = {err_bad:.1e},  'bound' P(Poisson(1) > N) = {poisson.sf(N, 1.0):.1e}")
for N in (10, 20, 40):
    print(f"max |P^{N}| = {np.abs(np.linalg.matrix_power(P_bad, N)).max():.3g}")
print(simulate_chain(P_bad, 0, 10, rng, n_paths=5))          # 예외 없음 — 상태 1 에 갇힌다
# -> diag (-0.5, -1, -1), False; 8.0e-06 (N=10), ~1e-15 (N=20); 460, 2.9e5, 8.1e10
```

</details>

## 7. 정리

1. $\lambda\ge\max q_i$면 $P=I+Q/\lambda$는 확률행렬이고 $e^{Qt}=\sum_ne^{-\lambda t}\frac{(\lambda t)^n}{n!}P^n$ — 시간(푸아송)과 공간(DTMC)의 분리, $P(t)=E[P^{N(t)}]$.
2. 절단 오차 $\le P(\mathrm{Poisson}(\lambda t)>N)$; $\lambda t=2$면 14항으로 $10^{-8}$. $\lambda$는 결과를 바꾸지 않고 항 수와 자기 루프 확률만 바꾼다.
3. 시뮬레이션: $N_i\sim\mathrm{Poisson}(\lambda t)$ 스텝의 DTMC를 벡터화해 돌리고 `X[np.arange(n), N_i]`를 읽으면 $X_t$의 분포가 정확히 $P(t)$의 행이다 — 질레스피보다 훨씬 빠르다.
4. 균일화 사슬의 정상분포는 $\pi$와 같고(점프 사슬 $\nu$와 다름), 자기 루프 덕에 비주기적 — 이것이 CTMC에 주기가 없는 이유(03b 정리 3의 빈칸).
5. $\lambda<\max q_i$면 확률 해석과 오차 한계가 사라지고, 큰 항의 상쇄로 수치도 불안정해진다.

**trap 카드**
- Q: 균일화 사슬 $P=I+Q/\lambda$와 점프 사슬 $\Pi$(03a) 중 어느 것의 정상분포가 CTMC의 $\pi$와 같은가?
- A: 균일화 사슬 $P$다: $\pi P=\pi+\pi Q/\lambda=\pi$. 점프 사슬은 자기 루프가 없어 '방문 횟수' 분포 $\nu$를 주고 $\pi_i\propto\nu_i/q_i$로 달라진다. $Q_3$: $\pi=(0.5,0.25,0.25)$, $\nu=(3/7,2/7,2/7)$.

**다음 노트북**: 03r — 모듈 02(푸아송)와 03(CTMC·대기행렬)을 닫힌 책으로 복습하고 세 개의 핵심 결과를 처음부터 재유도한 뒤, 캡스톤 C2(대기행렬 설계)로 간다.

`log/progress.md` 한 줄 템플릿:
`- [ ] 03f 균일화 — 날짜: ____ / 예측 적중: __/5 / E1 __ E2 __ E3 __ / 막힌 곳: ____`